# Prosody Emotion Training — RAVDESS (+ CREMA-D) → `prosody_emotion.npz`

Trains the built-in **`prosody-v1`** speech-emotion backend from scratch. No
pretrained models, no GPU, no torch — pure numpy, and the exact same feature
code runs at inference time inside the live avatar (train/serve parity by
construction).

**Pipeline**
1. Clone the project repo (or reuse the existing checkout) and make sure the
   prosody files are present — self-healing: embedded byte-identical copies
   are written when the repo does not contain them yet.
2. Download + extract **RAVDESS** speech audio (Zenodo, ~215 MB, 1,440 clips,
   24 actors — CC BY-NC-SA, non-commercial research use; cite Livingstone &
   Russo 2018).
3. Optionally download **CREMA-D** audio (git-LFS, ~600 MB, 7,442 clips,
   91 actors). Adds speaker diversity; skipped automatically if the download
   fails.
4. Extract the 56-dim prosodic features (optional 5x augmentation) and train a
   6-class softmax regression in numpy — speaker-grouped k-fold CV, then a
   final refit on 100% of the data.
5. Write the ~7 KB weights to `IMTalker/checkpoints/prosody_emotion.npz` and
   copy them to `/workspace/prosody_emotion.npz` for the streaming notebook.
6. *(Optional)* Batch-test any audio folder against the trained weights with
   confidence scores (Step 8).

**Runtime** on a RunPod CPU: ~25–40 min total (downloads ~5–10 min, feature
extraction ~15–25 min with augmentation, the training itself takes seconds).
A CPU-only pod is enough.

**Expected accuracy:** ~55–70% speaker-independent CV (prosody-only features
on acted speech). That is the honest range for hand-crafted prosodic features —
the win over the built-in rule-based fallback is consistency, not magic.

> **Companion notebook:** `runpod_emotion_streaming_prosody.ipynb` picks the
> trained weights up from `PROSODY_WEIGHTS_FILE` (`/workspace/prosody_emotion.npz`)
> automatically when run on the same pod. If you train elsewhere, upload the
> small `.npz` to the pod at that path (Jupyter file upload) before its Step 12.


In [ ]:
# --- Parameters (edit me) -----------------------------------------------------
REPO_URL = "https://github.com/MoshiHead/IMTalker_Personaplex_emotion_v1_add_prosody_emotion.git"
PROJECT_ROOT = "/workspace/speech2avatar"       # repo checkout (shared with the streaming notebook)
DATA_DIR = "/workspace/prosody_data"            # dataset downloads (kept outside the repo)
WEIGHTS_OUT = "/workspace/prosody_emotion.npz"  # hand-off path the streaming notebook reads
USE_CREMAD = True   # +7,442 clips / 91 speakers via git-lfs; auto-falls back to RAVDESS-only
AUGMENT = True      # 5x data: gain {0.7, 1.3} x tape-speed {0.9, 1.1}
FOLDS = 5           # speaker-grouped cross-validation folds
SEED = 0


## Step 1 — Clone / verify the project repo

Clones the prosody-emotion repo into `PROJECT_ROOT` (skipped when a checkout is
already present, e.g. shared with the streaming notebook). Then verifies the
two files this notebook needs:

- `IMTalker/prosody_emotion.py` — 56-dim feature extractor + classifier
- `scripts/train_prosody_emotion.py` — the training pipeline

If the repo does not contain them yet (fresh repo), **embedded copies** —
byte-identical to the source of truth — are written, with a warning. Push your
local workspace to the repo to make the embedded path unnecessary.


In [ ]:
import subprocess
from pathlib import Path

REQUIRED_FILES = ["IMTalker/prosody_emotion.py", "scripts/train_prosody_emotion.py"]

# Embedded byte-identical copies, written only when the repo lacks them.
EMBEDDED = {
    "IMTalker/prosody_emotion.py": r'''"""Lightweight prosody-based speech emotion detection (PROSODY_SER_V1).

Replaces the emotion2vec+/funasr SER backend with a from-scratch, pure-numpy
pipeline that needs no pretrained model, no GPU and no downloads:

  utterance (24 kHz float32 mono, from auto_emotion.EnergyVAD)
    -> 25 ms frames / 20 ms hop (Hann windowed)
    -> per-frame contours: RMS(dB), F0 (autocorrelation), ZCR, spectral
       centroid / rolloff(85%) / flatness / flux, 13 MFCCs (precomputed mel
       filterbank + DCT)
    -> utterance-level statistics per contour (mean/std/min/max/range/slope)
       plus voiced-ratio, syllable rate, pause ratio, jitter, shimmer,
       duration and MFCC1-4 mean/std  ->  ~56-dim feature vector
    -> z-score normalise -> 6-class softmax linear model (one matrix
       multiply) -> (emotion_id, label, confidence)

Cost: ~2-5 ms CPU per utterance, zero VRAM, zero torch.

Classifier contract (identical to auto_emotion.SpeechEmotionClassifier.predict
so the manager / server patches / notebook smoke test work unchanged):

    predict(pcm_24k) -> (emotion_id: int 0-5, label: str, confidence: float)

Two backends share the same feature vector and softmax machinery:
  * "softmax" - weights trained from scratch on RAVDESS (+ optionally
    CREMA-D) by scripts/train_prosody_emotion.py and stored in a ~10 KB
    .npz (weights, bias, feature mean/std, feature names, temperature).
  * "rules"   - hand-crafted z-space weight matrix + population priors,
    used automatically when no .npz is found. Works for any speaker at
    moderate accuracy; the trained model replaces it transparently.

Weight-file search order: explicit path -> $PROSODY_EMOTION_WEIGHTS ->
<IMTalker>/checkpoints/prosody_emotion.npz -> <IMTalker>/prosody_emotion.npz
-> rule-based fallback (with a one-time guidance print).
"""

from __future__ import annotations

import os
from pathlib import Path

import numpy as np

# Emotion ids must match generator/FM.py EMOTION_MAP and auto_emotion.py.
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
NUM_EMOTIONS = 6

INPUT_SR = 24_000          # matches auto_emotion.TARGET_SR (server mic PCM rate)
FRAME_LEN = 600            # 25 ms @ 24 kHz
HOP = 480                  # 20 ms hop (utterance-level stats need no finer grid;
                           # halves the FFT count vs 10 ms, aligns with the VAD frame)
NFFT = 2048                # zero-padded rFFT, shared by the spectral features and
                           # the autocorrelation pitch track (one FFT serves both)
F0_MIN_HZ = 60.0           # ACF search range
F0_MAX_HZ = 400.0
LAG_MIN = int(round(INPUT_SR / F0_MAX_HZ))   # 60
LAG_MAX = int(round(INPUT_SR / F0_MIN_HZ))   # 400
VOICED_ACF_MIN = 0.35      # normalised-ACF peak strength to call a frame voiced
NUM_MEL = 26
NUM_MFCC = 13
MEL_FMAX = 8000.0
SILENCE_DB = -60.0         # whole-utterance guard: below this -> no speech
TARGET_RMS = 0.1           # loudness-normalisation target (~-20 dBFS RMS):
                           # recording level is NOT an emotion cue. Without this
                           # normalisation the level-dependent features
                           # (rms_db_*, mfcc1_*, shimmer) go out-of-distribution
                           # for any non-studio-level input and the softmax
                           # collapses to one class for every file.
RMS_SILENCE = 10.0 ** (SILENCE_DB / 20.0)  # -60 dBFS utterance-RMS guard
Z_CLIP = 5.0               # winsorise standardised features: legit training
                           # variation is |z| <~ 3.5; clipping bounds the logit
                           # contribution of any out-of-distribution feature so
                           # it can never saturate the softmax (conf 1.00 on
                           # every input) again. In-distribution predictions
                           # are untouched.
WEIGHTS_FORMAT_VERSION = 3  # v3 = edge-trimmed + loudness-normalised features.
                           # v1/v2 weight files are incompatible (different
                           # feature semantics) and are rejected with a retrain
                           # hint.

# ---------------------------------------------------------------------------
# Feature layout (kept in ONE place; the trainer validates against this list)
# ---------------------------------------------------------------------------

_STAT_CONTOURS = ("rms_db", "f0", "zcr", "centroid", "rolloff", "flatness", "flux")
_STAT_NAMES = ("mean", "std", "min", "max", "range", "slope")
_EXTRA_NAMES = ("voiced_ratio", "syll_rate", "pause_ratio", "jitter", "shimmer", "duration")
_MFCC_STAT_NAMES = tuple(f"mfcc{k}_{s}" for k in (1, 2, 3, 4) for s in ("mean", "std"))

FEATURE_NAMES = (
    tuple(f"{c}_{s}" for c in _STAT_CONTOURS for s in _STAT_NAMES)
    + _EXTRA_NAMES
    + _MFCC_STAT_NAMES
)
FEATURE_DIM = len(FEATURE_NAMES)
_FEATURE_INDEX = {name: i for i, name in enumerate(FEATURE_NAMES)}



# ---------------------------------------------------------------------------
# Lazy precomputed tables (built once per process)
# ---------------------------------------------------------------------------

_TABLES: dict = {}


def _tables() -> dict:
    if _TABLES:
        return _TABLES
    win = np.hanning(FRAME_LEN).astype(np.float32)
    freqs = np.fft.rfftfreq(NFFT, 1.0 / INPUT_SR).astype(np.float64)

    # Mel filterbank (linearly spaced in mel, triangular in Hz)
    def hz_to_mel(f):
        return 2595.0 * np.log10(1.0 + np.asarray(f, dtype=np.float64) / 700.0)

    def mel_to_hz(m):
        return 700.0 * (10.0 ** (np.asarray(m, dtype=np.float64) / 2595.0) - 1.0)

    mel_pts = np.linspace(hz_to_mel(0.0), hz_to_mel(MEL_FMAX), NUM_MEL + 2)
    hz_pts = mel_to_hz(mel_pts)
    bin_pts = np.floor((NFFT + 1) * hz_pts / INPUT_SR).astype(int)
    mel_fb = np.zeros((NUM_MEL, NFFT // 2 + 1), dtype=np.float32)
    for m in range(NUM_MEL):
        lo, ctr, hi = bin_pts[m], bin_pts[m + 1], bin_pts[m + 2]
        if ctr <= lo:
            ctr = lo + 1
        if hi <= ctr:
            hi = ctr + 1
        for k in range(lo, min(ctr, mel_fb.shape[1])):
            mel_fb[m, k] = (k - lo) / float(ctr - lo)
        for k in range(ctr, min(hi, mel_fb.shape[1])):
            mel_fb[m, k] = (hi - k) / float(hi - ctr)

    # DCT-II matrix for MFCCs
    n = np.arange(NUM_MEL)
    k = np.arange(NUM_MFCC)[:, None]
    dct = np.cos(np.pi / NUM_MEL * (n + 0.5) * k).astype(np.float32)

    _TABLES.update(win=win, freqs=freqs, mel_fb=mel_fb, dct=dct)
    return _TABLES


# ---------------------------------------------------------------------------
# Feature extraction (pure numpy, vectorised; ~2-5 ms per 3 s utterance)
# ---------------------------------------------------------------------------

def _contour_stats(vals: dict, name: str, v: np.ndarray, times: np.ndarray) -> None:
    """Closed-form mean/std/min/max/range/slope(per-second) for one contour."""
    m = int(v.shape[0])
    if m < 2:
        for s in _STAT_NAMES:
            vals[f"{name}_{s}"] = 0.0
        return
    v = np.asarray(v, dtype=np.float64)
    t = times[:m].astype(np.float64)
    tc = t - t.mean()
    denom = float(np.dot(tc, tc))
    slope = float(np.dot(tc, v - v.mean()) / denom) if denom > 0 else 0.0
    vals[f"{name}_mean"] = float(v.mean())
    vals[f"{name}_std"] = float(v.std())
    vals[f"{name}_min"] = float(v.min())
    vals[f"{name}_max"] = float(v.max())
    vals[f"{name}_range"] = float(v.max() - v.min())
    vals[f"{name}_slope"] = slope


EDGE_TRIM_RANGE_DB = 35.0              # frames within this many dB of the peak
                                       # frame count as speech at the edges
EDGE_TRIM_FLOOR_DB = SILENCE_DB + 10.0  # absolute floor (-50 dBFS): below this
                                        # a frame is silence no matter what


def trim_edges(pcm: np.ndarray) -> np.ndarray:
    """Trim leading/trailing (near-)silence from a waveform.

    RAVDESS/CREMA-D clips are studio-trimmed; real recordings carry 0.5-2 s of
    edge silence or fade-outs, which corrupts duration, rms_db stats/slopes,
    flatness_slope and pause features (proven: padding a clean profile flips
    its prediction and reproduces the exact out-of-distribution signature seen
    on user audio). Only the edges are trimmed - internal pauses are preserved.
    Returns an empty array when nothing is above the threshold.
    """
    x = np.asarray(pcm, dtype=np.float32).reshape(-1)
    if x.shape[0] < FRAME_LEN * 2:
        return x
    sw = np.lib.stride_tricks.sliding_window_view(x, FRAME_LEN)[::HOP]
    fr = np.sqrt(np.mean(np.square(sw, dtype=np.float32), axis=1))
    fr_db = 20.0 * np.log10(fr + 1e-10)
    thr = max(float(fr_db.max()) - EDGE_TRIM_RANGE_DB, EDGE_TRIM_FLOOR_DB)
    above = np.nonzero(fr_db > thr)[0]
    if above.size == 0:
        return x[:0]
    return x[above[0] * HOP: above[-1] * HOP + FRAME_LEN]


def extract_features(pcm: np.ndarray) -> np.ndarray:
    """24 kHz float32 mono utterance -> FEATURE_NAMES-ordered feature vector.

    Returns an all-zero vector for degenerate input (too short / fully
    silent); callers treat that as "no prediction possible".
    """
    tb = _tables()
    x = np.asarray(pcm, dtype=np.float32).reshape(-1)
    n = int(x.shape[0])
    zeros = np.zeros(FEATURE_DIM, dtype=np.float32)
    if n < FRAME_LEN * 2:
        return zeros

    # VAD edge trim: real-world files carry leading/trailing silence that the
    # studio training data never shows (see trim_edges). Idempotent no-op on
    # already-trimmed input (RAVDESS, live VAD utterances).
    x = trim_edges(x)
    n = int(x.shape[0])
    if n < FRAME_LEN * 2:
        return zeros

    # Utterance-level silence guard (un-normalised signal), then loudness
    # normalisation so rms_db/mfcc/shimmer features are level-invariant.
    u_rms = float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))
    if u_rms < RMS_SILENCE:
        return zeros
    x = (x * (TARGET_RMS / u_rms)).astype(np.float32)

    sw = np.lib.stride_tricks.sliding_window_view(x, FRAME_LEN)
    frames = sw[::HOP]                      # (n_frames, FRAME_LEN), a view
    nf = int(frames.shape[0])
    if nf < 2:
        return zeros
    fw = frames * tb["win"]

    # --- energy -------------------------------------------------------------
    rms = np.sqrt(np.mean(np.square(frames, dtype=np.float32), axis=1))
    rms_db = 20.0 * np.log10(rms + 1e-8)
    if float(rms_db.max()) < SILENCE_DB:
        return zeros

    # --- spectrum (one rFFT shared by the spectral features and the F0 ACF) --
    A = np.fft.rfft(fw, n=NFFT, axis=1)                      # (nf, 1025)
    mag = np.abs(A)
    pow_ = np.square(mag)
    msum = mag.sum(axis=1) + 1e-10
    centroid = (mag @ tb["freqs"]) / msum
    csum = np.cumsum(mag, axis=1)
    roll_idx = (csum >= (0.85 * msum)[:, None]).argmax(axis=1)
    rolloff = tb["freqs"][roll_idx]
    flatness = np.exp(np.mean(np.log(mag + 1e-10), axis=1)) / (np.mean(mag, axis=1) + 1e-10)
    dmag = np.diff(mag, axis=0)
    flux = np.concatenate([[0.0], np.sqrt(np.mean(np.square(dmag), axis=1))])
    zcr = np.mean(frames[:, 1:] * frames[:, :-1] < 0, axis=1)

    # --- F0 via normalised autocorrelation (vectorised over frames) ---------
    acf = np.fft.irfft(pow_, n=NFFT, axis=1)[:, :LAG_MAX + 1]
    acf_n = acf / (acf[:, :1] + 1e-10)
    seg = acf_n[:, LAG_MIN:LAG_MAX + 1]
    peak_lag = seg.argmax(axis=1) + LAG_MIN
    peak_val = seg.max(axis=1)
    voiced = (peak_val > VOICED_ACF_MIN) & (rms_db > (float(rms_db.mean()) - 10.0))
    f0 = np.where(voiced, INPUT_SR / peak_lag.astype(np.float32), 0.0)
    f0v = f0[f0 > 0.0]

    # --- MFCCs ---------------------------------------------------------------
    mel_pow = tb["mel_fb"] @ pow_.T                          # (26, nf)
    mfcc = tb["dct"] @ np.log(mel_pow + 1e-10)               # (13, nf)

    # --- utterance-level statistics ------------------------------------------
    times = np.arange(nf, dtype=np.float32) * (HOP / INPUT_SR)
    vals: dict = {}
    _contour_stats(vals, "rms_db", rms_db, times)
    _contour_stats(vals, "f0", f0v, times)        # voiced frames only
    _contour_stats(vals, "zcr", zcr, times)
    _contour_stats(vals, "centroid", centroid, times)
    _contour_stats(vals, "rolloff", rolloff, times)
    _contour_stats(vals, "flatness", flatness, times)
    _contour_stats(vals, "flux", flux, times)

    duration = n / float(INPUT_SR)
    vals["voiced_ratio"] = float(f0v.shape[0]) / float(nf)

    # syllable rate: peaks of the smoothed RMS contour (>=120 ms apart)
    kernel = np.ones(5, dtype=np.float32) / 5.0
    rms_s = np.convolve(rms, kernel, mode="same")
    thresh = float(rms_s.mean() + 0.5 * rms_s.std())
    cand = np.where(
        (rms_s[1:-1] > thresh)
        & (rms_s[1:-1] >= rms_s[:-2])
        & (rms_s[1:-1] > rms_s[2:])
    )[0] + 1
    min_dist = max(1, int(round(0.12 / (HOP / INPUT_SR))))
    peaks: list = []
    last = -min_dist
    for i in cand:
        i = int(i)
        if i - last >= min_dist:
            peaks.append(i)
            last = i
        elif peaks and rms_s[i] > rms_s[peaks[-1]]:
            peaks[-1] = i
            last = i
    vals["syll_rate"] = float(len(peaks)) / max(duration, 1e-6)

    vals["pause_ratio"] = float(np.mean(rms_db < (float(rms_db.mean()) - 8.0)))
    if f0v.shape[0] >= 3:
        vals["jitter"] = float(np.abs(np.diff(f0v)).mean() / (f0v.mean() + 1e-6))
    else:
        vals["jitter"] = 0.0
    vals["shimmer"] = float(np.abs(np.diff(rms)).mean() / (rms.mean() + 1e-8))
    vals["duration"] = float(duration)
    for k in (1, 2, 3, 4):
        vals[f"mfcc{k}_mean"] = float(mfcc[k].mean())
        vals[f"mfcc{k}_std"] = float(mfcc[k].std())

    out = np.array([vals[name] for name in FEATURE_NAMES], dtype=np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)



# ---------------------------------------------------------------------------
# Softmax linear model (shared by the trained backend and the rule fallback)
# ---------------------------------------------------------------------------

class _SoftmaxEmotionModel:
    """z-scored features -> 6-class softmax. Inference is one matmul."""

    def __init__(self, W: np.ndarray, b: np.ndarray, feat_mean: np.ndarray,
                 feat_std: np.ndarray, temperature: float = 1.0) -> None:
        self.W = np.asarray(W, dtype=np.float32)
        self.b = np.asarray(b, dtype=np.float32)
        self.feat_mean = np.asarray(feat_mean, dtype=np.float32)
        self.feat_std = np.asarray(feat_std, dtype=np.float32)
        self.temperature = float(temperature)

    def predict_proba(self, feats: np.ndarray) -> np.ndarray:
        z = (np.asarray(feats, dtype=np.float32) - self.feat_mean) / self.feat_std
        z = np.nan_to_num(z, nan=0.0, posinf=Z_CLIP, neginf=-Z_CLIP)
        z = np.clip(z, -Z_CLIP, Z_CLIP)
        logits = (self.W @ z + self.b) / max(self.temperature, 1e-3)
        logits = logits - logits.max()
        e = np.exp(logits)
        return (e / e.sum()).astype(np.float32)


# ---------------------------------------------------------------------------
# Rule-based fallback: population priors + hand-crafted z-space weights.
# Rationale per emotion (prosody literature on acted/emotional speech):
#   happy    - high/wide F0, fast rate, rising pitch
#   sad      - low/flat F0, quiet, slow, more pauses
#   angry    - loud, noisy (high ZCR/flux), tense (jitter), bright spectrum
#   surprise - high F0 with a strong rising contour, short duration
#   fear     - high jitter/shimmer (tremor), breathy (flatness), tense
#   neutral  - the flat middle ground (negative weights on variability)
# ---------------------------------------------------------------------------

# Priors for the level-dependent features are calibrated to the
# loudness-normalised (v2) feature space: after normalisation every utterance
# has RMS 0.1, pinning rms_db_mean near -23 dB and flux near 0.5 (measured on
# the trainer's synthetic profiles). The old un-normalised priors
# (rms_db -30 dB, flux 0.03) would give every input the same large angry-biased
# z-scores.
_BASE_PRIORS = {
    "rms_db": (-23.0, 6.0), "f0": (150.0, 60.0), "zcr": (0.12, 0.05),
    "centroid": (1600.0, 500.0), "rolloff": (3200.0, 1000.0),
    "flatness": (0.06, 0.03), "flux": (0.5, 0.25),
}
_EXTRA_PRIORS = {
    "voiced_ratio": (0.75, 0.10), "syll_rate": (4.0, 1.5), "pause_ratio": (0.15, 0.10),
    "jitter": (0.02, 0.012), "shimmer": (0.08, 0.04), "duration": (3.0, 1.5),
}

_RULE_WEIGHTS = {
    0: {"f0_std": -0.5, "rms_db_std": -0.3, "jitter": -0.4, "zcr_std": -0.3, "flux_mean": -0.3},
    1: {"f0_mean": 0.9, "f0_std": 0.8, "f0_range": 0.5, "f0_slope": 0.4,
        "rms_db_mean": 0.4, "syll_rate": 0.5, "flux_mean": 0.3},
    2: {"f0_mean": -1.0, "rms_db_mean": -0.8, "syll_rate": -0.8, "f0_std": -0.6,
        "f0_slope": -0.5, "flux_mean": -0.3, "pause_ratio": 0.4, "duration": 0.2},
    3: {"rms_db_mean": 1.0, "f0_std": 0.7, "zcr_mean": 0.7, "flux_mean": 0.6,
        "jitter": 0.4, "f0_mean": 0.4, "centroid_mean": 0.4},
    4: {"f0_mean": 0.9, "f0_slope": 0.8, "f0_range": 0.6, "f0_max": 0.4,
        "duration": -0.4, "rms_db_max": 0.3},
    5: {"f0_mean": 0.5, "jitter": 1.0, "shimmer": 0.9, "flatness_mean": 0.4,
        "zcr_mean": 0.3, "syll_rate": 0.3},
}
_RULE_BIAS = {0: 0.5}  # mild prior toward neutral (the conversational default)


def _prior_for(feature_name: str) -> tuple:
    for contour, (mu, sigma) in _BASE_PRIORS.items():
        if feature_name.startswith(contour + "_"):
            stat = feature_name[len(contour) + 1:]
            if stat == "mean":
                return mu, max(sigma * 0.6, 1e-6)
            if stat == "std":
                return sigma, max(sigma * 0.6, 1e-6)
            if stat == "min":
                return mu - sigma, max(sigma, 1e-6)
            if stat == "max":
                return mu + sigma, max(sigma, 1e-6)
            if stat == "range":
                return 2.0 * sigma, max(sigma, 1e-6)
            return 0.0, max(sigma * 0.5, 1e-6)  # slope
    if feature_name in _EXTRA_PRIORS:
        mu, sigma = _EXTRA_PRIORS[feature_name]
        return mu, max(sigma, 1e-6)
    if feature_name.startswith("mfcc"):
        # cosmetic only - no rule weights touch the mfcc features
        return (40.0, 12.0) if feature_name.endswith("_mean") else (10.0, 4.0)
    return 0.0, 1.0


def _build_rule_model() -> _SoftmaxEmotionModel:
    mean = np.zeros(FEATURE_DIM, dtype=np.float32)
    std = np.ones(FEATURE_DIM, dtype=np.float32)
    for i, name in enumerate(FEATURE_NAMES):
        mean[i], std[i] = _prior_for(name)
    W = np.zeros((NUM_EMOTIONS, FEATURE_DIM), dtype=np.float32)
    b = np.zeros(NUM_EMOTIONS, dtype=np.float32)
    for eid, weights in _RULE_WEIGHTS.items():
        for name, w in weights.items():
            if name in _FEATURE_INDEX:
                W[eid, _FEATURE_INDEX[name]] = w
    for eid, bias in _RULE_BIAS.items():
        b[eid] = bias
    return _SoftmaxEmotionModel(W, b, mean, std, temperature=1.5)


_RULE_MODEL = _build_rule_model()



# ---------------------------------------------------------------------------
# Weight file I/O (shared with scripts/train_prosody_emotion.py)
# ---------------------------------------------------------------------------

def save_weights(path, W: np.ndarray, b: np.ndarray, feat_mean: np.ndarray,
                 feat_std: np.ndarray, temperature: float = 1.0,
                 extra_files=None) -> None:
    """Persist a trained model (~7 KB npz).

    extra_files: optional list of user file names that were part of training
    (trainer --extra_dir), stored as metadata so the batch-test cell can flag
    predictions on those files as training-set results.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        path,
        W=np.asarray(W, dtype=np.float32),
        b=np.asarray(b, dtype=np.float32),
        feat_mean=np.asarray(feat_mean, dtype=np.float32),
        feat_std=np.asarray(feat_std, dtype=np.float32),
        feature_names=np.array(FEATURE_NAMES),
        temperature=np.float32(temperature),
        format_version=np.int32(WEIGHTS_FORMAT_VERSION),
        extra_files=np.array(list(extra_files or []), dtype=str),
    )


def load_weights(path) -> _SoftmaxEmotionModel:
    data = np.load(path, allow_pickle=False)
    version = int(data["format_version"]) if "format_version" in data.files else 1
    if version != WEIGHTS_FORMAT_VERSION:
        raise ValueError(
            f"weight file {path} is format v{version} but this prosody_emotion.py "
            f"expects v{WEIGHTS_FORMAT_VERSION} (loudness-normalised features). "
            f"Retrain with scripts/train_prosody_emotion.py (or re-run the "
            f"training notebook) to regenerate prosody_emotion.npz."
        )
    names = [str(x) for x in data["feature_names"]]
    if tuple(names) != FEATURE_NAMES:
        raise ValueError(
            f"weight file {path} feature layout does not match this "
            f"prosody_emotion.py (retrain with scripts/train_prosody_emotion.py)"
        )
    return _SoftmaxEmotionModel(
        data["W"], data["b"], data["feat_mean"], data["feat_std"],
        float(data["temperature"]),
    )


# ---------------------------------------------------------------------------
# Public classifier (contract-compatible with SpeechEmotionClassifier.predict)
# ---------------------------------------------------------------------------

_MODULE_DIR = Path(__file__).resolve().parent


class ProsodyEmotionClassifier:
    """predict(pcm_24k) -> (emotion_id, label, confidence). CPU-only."""

    def __init__(self, weights_path=None, verbose: bool = True) -> None:
        candidates = []
        if weights_path:
            candidates.append(Path(weights_path))
        env_path = os.environ.get("PROSODY_EMOTION_WEIGHTS", "").strip()
        if env_path:
            candidates.append(Path(env_path))
        candidates.append(_MODULE_DIR / "checkpoints" / "prosody_emotion.npz")
        candidates.append(_MODULE_DIR / "prosody_emotion.npz")

        self.model: _SoftmaxEmotionModel | None = None
        self.backend = "rules"
        self.weights_path: Path | None = None
        for cand in candidates:
            try:
                if cand.is_file():
                    self.model = load_weights(cand)
                    self.backend = "softmax"
                    self.weights_path = cand
                    break
            except Exception as exc:
                print(f"[PROSODY-SER] failed to load {cand}: {exc!r}; trying next", flush=True)
        if self.model is None:
            self.model = _RULE_MODEL
            if verbose:
                print(
                    "[PROSODY-SER] no trained prosody_emotion.npz found - using the "
                    "rule-based fallback. Train the tiny softmax on RAVDESS with "
                    "scripts/train_prosody_emotion.py for better accuracy.",
                    flush=True,
                )
        elif verbose:
            print(f"[PROSODY-SER] loaded trained weights: {self.weights_path}", flush=True)

    def predict(self, pcm_24k: np.ndarray):
        """24 kHz float32 mono utterance -> (emotion_id, label, confidence)."""
        feats = extract_features(pcm_24k)
        if not feats.any():
            return 0, "neutral", 0.0
        probs = self.model.predict_proba(feats)
        eid = int(np.argmax(probs))
        return eid, EMOTION_NAMES[eid], float(probs[eid])

''',
    "scripts/train_prosody_emotion.py": r'''#!/usr/bin/env python3
"""Train the tiny prosody softmax classifier (PROSODY_SER_V1) from scratch.

No pretrained models anywhere: features come from prosody_emotion.extract_features
(the exact same function used at inference, so train/serve parity is guaranteed)
and the classifier is a multinomial logistic regression trained with plain numpy
gradient descent. The result is a ~7 KB .npz that prosody_emotion.py loads
transparently (replacing its rule-based fallback).

Datasets (download manually, they are public but not pip-installable):
  * RAVDESS speech audio (Zenodo, Audio_Speech_Actors_01-24.zip): 24 actors,
    1,440 clips, 8 emotion categories. Mapping to the avatar ids:
    01 neutral -> 0, 02 calm -> 0, 03 happy -> 1, 04 sad -> 2, 05 angry -> 3,
    06 fearful -> 5, 07 disgust -> skipped (the emotion LoRA has no disgust),
    08 surprised -> 4.
  * CREMA-D (optional, its GitHub repo provides the wav archive): 91 actors,
    7,442 clips. ANG -> 3, FEA -> 5, HAP -> 1, NEU -> 0, SAD -> 2 (DIS skipped;
    CREMA-D has no surprise class).

Validation is speaker-grouped k-fold CV: no speaker appears in both train and
test, so the reported accuracy is an honest estimate for a *new* voice.

Channel robustness: --noise_aug (default ON) adds a noisy copy of every
training utterance (white/pink noise, random 6-30 dB SNR). Without it the
studio-trained model learns "bright/noisy spectrum = angry" and collapses
every real-world (mic + room noise) recording to angry - reproduced
experimentally: at 20 dB SNR all clean-trained predictions flip to angry.

--extra_dir accepts a user folder with files named <emotion>-*.wav
(angry/fear/happy/sad/neutral/surprise), added to training x10 so a handful of
files matters. They form their own CV speaker group, so the CV report shows an
honest held-out accuracy on the user's audio.

Usage:
  python scripts/train_prosody_emotion.py --ravdess_dir /data/ravdess \
      [--cremad_dir /data/cremad] [--augment] [--extra_dir /workspace/audio] \
      [--no_noise_aug] [--out IMTalker/checkpoints/prosody_emotion.npz]
  python scripts/train_prosody_emotion.py --selftest --out /tmp/prosody.npz

After training, re-run scripts/switch_ser_to_prosody.py to embed the new
weights into runpod_emotion_streaming.ipynb (Step 12 writes them onto the pod).
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT / "IMTalker"))

import prosody_emotion as pe  # noqa: E402

SR = pe.INPUT_SR

# RAVDESS emotion-code -> avatar id (None = skip the sample)
RAVDESS_MAP = {"01": 0, "02": 0, "03": 1, "04": 2, "05": 3, "06": 5, "07": None, "08": 4}
# CREMA-D emotion-code -> avatar id (None = skip)
CREMAD_MAP = {"ANG": 3, "DIS": None, "FEA": 5, "HAP": 1, "NEU": 0, "SAD": 2}


# ---------------------------------------------------------------------------
# Audio loading (soundfile -> scipy -> stdlib wave fallback chain)
# ---------------------------------------------------------------------------

def _resample_to(x: np.ndarray, src_sr: int, dst_sr: int) -> np.ndarray:
    if src_sr == dst_sr:
        return x.astype(np.float32)
    try:
        from scipy.signal import resample_poly
        from math import gcd
        g = gcd(int(src_sr), int(dst_sr))
        return resample_poly(x, dst_sr // g, src_sr // g).astype(np.float32)
    except Exception:
        t_old = np.linspace(0, len(x) / src_sr, num=len(x), endpoint=False)
        n_new = int(round(len(x) * dst_sr / src_sr))
        t_new = np.linspace(0, len(x) / src_sr, num=n_new, endpoint=False)
        return np.interp(t_new, t_old, x).astype(np.float32)


def load_wav(path: Path, target_sr: int = SR) -> np.ndarray:
    try:
        import soundfile as sf
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        return _resample_to(data.mean(axis=1), sr, target_sr)
    except ImportError:
        pass
    try:
        from scipy.io import wavfile
        sr, data = wavfile.read(str(path))
        if data.dtype == np.int16:
            data = data.astype(np.float32) / 32768.0
        elif data.dtype == np.int32:
            data = data.astype(np.float32) / 2147483648.0
        else:
            data = np.asarray(data, dtype=np.float32)
        if data.ndim > 1:
            data = data.mean(axis=1)
        return _resample_to(data, sr, target_sr)
    except ImportError:
        pass
    import wave
    with wave.open(str(path), "rb") as w:
        sr = w.getframerate()
        n_ch = w.getnchannels()
        raw = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    data = raw.astype(np.float32) / 32768.0
    if n_ch > 1:
        data = data.reshape(-1, n_ch).mean(axis=1)
    return _resample_to(data, sr, target_sr)


# ---------------------------------------------------------------------------
# Channel-noise augmentation (the real-world robustness fix).
# RAVDESS/CREMA-D are studio recordings; real audio carries mic/room noise.
# Without a noisy copy of every training utterance, the noise direction
# (high zcr/rolloff/mfcc-std/flux) stays discriminative and points at angry,
# collapsing non-studio input to one class. Adding noisy copies makes those
# features non-discriminative and forces the model onto noise-robust prosodic
# cues (f0, syllable rate, energy dynamics).
# ---------------------------------------------------------------------------

NOISE_SNR_RANGE = (6.0, 30.0)   # dB, uniform per augmented copy


def add_channel_noise(x: np.ndarray, rng: np.random.Generator,
                      snr_db: float | None = None) -> np.ndarray:
    """One noisy copy of x: white or pink noise at the given/random SNR."""
    p_sig = float(np.sqrt(np.mean(np.square(x, dtype=np.float64)))) + 1e-12
    snr = float(rng.uniform(*NOISE_SNR_RANGE)) if snr_db is None else float(snr_db)
    noise = rng.standard_normal(x.shape[0])
    if rng.random() < 0.5:          # pink-ish: integrated white, closer to room noise
        noise = np.cumsum(noise)
        noise -= noise.mean()
    noise = noise / (float(np.sqrt(np.mean(np.square(noise)))) + 1e-12)
    return (x + noise * p_sig / 10 ** (snr / 20)).astype(np.float32)


# ---------------------------------------------------------------------------
# Dataset ingestion
# ---------------------------------------------------------------------------

# user extra-dir emotion prefix -> avatar id (None = skip the file)
EXTRA_MAP = {"neutral": 0, "calm": 0, "happy": 1, "sad": 2, "angry": 3,
             "surprise": 4, "surprised": 4, "fear": 5, "fearful": 5}
EXTRA_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".m4a"}
EXTRA_REPEAT = 10   # a few user files must matter among thousands of dataset clips


def scan_extra(root: Path):
    """Yield (wav_path, emotion_id, speaker_id) for user files named
    <emotion>-<anything>.<ext>, e.g. angry-1.wav. Unknown emotions are skipped."""
    out = []
    for wav in sorted(Path(root).rglob("*")):
        if wav.suffix.lower() not in EXTRA_EXTS:
            continue
        emo = EXTRA_MAP.get(wav.stem.split("-")[0].lower())
        if emo is None:
            continue
        out.append((wav, emo, "user_extra"))
    return out


def scan_ravdess(root: Path):
    """Yield (wav_path, emotion_id, speaker_id) for every usable RAVDESS clip."""
    out = []
    for wav in sorted(Path(root).rglob("*.wav")):
        parts = wav.stem.split("-")
        if len(parts) != 7:
            continue
        emo = RAVDESS_MAP.get(parts[2])
        if emo is None:
            continue
        out.append((wav, emo, f"rav_{parts[-1]}"))
    return out


def scan_cremad(root: Path):
    """Yield (wav_path, emotion_id, speaker_id) for every usable CREMA-D clip."""
    out = []
    for wav in sorted(Path(root).rglob("*.wav")):
        parts = wav.stem.split("_")
        if len(parts) < 3:
            continue
        emo = CREMAD_MAP.get(parts[2].upper())
        if emo is None:
            continue
        out.append((wav, emo, f"cre_{parts[0]}"))
    return out


def build_dataset(samples, augment: bool = False, noise_aug: bool = True,
                  rng: np.random.Generator | None = None):
    """samples: list of (wav_path, emotion_id, speaker_id).

    Returns X (n, FEATURE_DIM), y (n,), speakers (n,). Feature extraction uses
    the exact inference-time function, guaranteeing train/serve parity.
    Augmentation (optional, x5): gain {0.7, 1.3} and tape-style speed factors
    {0.9, 1.1} applied via resampling (pitch+tempo shift). With noise_aug
    (default on) every utterance also contributes one noisy copy (white/pink
    noise at a random 6-30 dB SNR) - the channel-robustness fix.
    """
    variants = [(1.0, 1.0)]
    if augment:
        variants += [(1.3, 1.0), (0.7, 1.0), (1.0, 1.1), (1.0, 0.9)]
    if rng is None:
        rng = np.random.default_rng(0)
    X, y, spk = [], [], []
    t0 = time.time()
    for i, (wav, emo, speaker) in enumerate(samples):
        try:
            base = load_wav(wav)
        except Exception as exc:
            print(f"  [warn] skipping {wav.name}: {exc!r}")
            continue
        for gain, speed in variants:
            x = base
            if speed != 1.0:
                t_old = np.linspace(0, len(x) / SR, num=len(x), endpoint=False)
                t_new = np.linspace(0, len(x) / SR / speed, num=int(len(x) / speed), endpoint=False)
                x = np.interp(t_new, t_old, x).astype(np.float32)
            if gain != 1.0:
                x = np.clip(x * gain, -1.0, 1.0)
            feats = pe.extract_features(x)
            if not feats.any():
                continue
            X.append(feats)
            y.append(emo)
            spk.append(speaker)
        if noise_aug:
            feats = pe.extract_features(add_channel_noise(base, rng))
            if feats.any():
                X.append(feats)
                y.append(emo)
                spk.append(speaker)
        if (i + 1) % 100 == 0:
            rate = (i + 1) / max(time.time() - t0, 1e-6)
            print(f"  features: {i + 1}/{len(samples)} files ({rate:.0f}/s)")
    return np.stack(X), np.array(y), np.array(spk)


# ---------------------------------------------------------------------------
# Softmax regression, from scratch (numpy full-batch gradient descent)
# ---------------------------------------------------------------------------

def train_softmax(Z: np.ndarray, y: np.ndarray, n_classes: int = pe.NUM_EMOTIONS,
                  iters: int = 1000, lr: float = 0.5, l2: float = 1e-4,
                  seed: int = 0):
    """Class-weighted multinomial logistic regression. Z is pre-standardised."""
    n, d = Z.shape
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.01, size=(n_classes, d))
    b = np.zeros(n_classes)
    Y = np.eye(n_classes)[y]
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    cw = n / (n_classes * np.maximum(counts, 1))       # balanced classes
    sw = cw[y][:, None]
    for it in range(iters):
        logits = Z @ W.T + b
        logits -= logits.max(axis=1, keepdims=True)
        P = np.exp(logits)
        P /= P.sum(axis=1, keepdims=True)
        E = (P - Y) * sw
        W -= lr * (E.T @ Z / n + l2 * W)
        b -= lr * E.mean(axis=0)
        if (it + 1) % 250 == 0:
            lr *= 0.5
    return W, b


def evaluate(W, b, Z, y):
    logits = Z @ W.T + b
    pred = logits.argmax(axis=1)
    return float((pred == y).mean()), pred


def speaker_cv(X, y, speakers, folds: int = 5, seed: int = 0):
    """Speaker-grouped k-fold CV. Returns (mean_acc, out_of_fold_preds, fold_accs)."""
    uniq = np.unique(speakers)
    rng = np.random.default_rng(seed)
    rng.shuffle(uniq)
    groups = [uniq[i::folds] for i in range(folds)]
    oof = np.full(len(y), -1)
    fold_accs = []
    for k in range(folds):
        test_mask = np.isin(speakers, groups[k])
        train_mask = ~test_mask
        mu = X[train_mask].mean(0)
        # 1e-3 floor (not 1e-6): after loudness normalisation some features are
        # near-constant, and a near-zero std would make z-scores unstable.
        sd = np.maximum(X[train_mask].std(0), 1e-3)
        W, b = train_softmax((X[train_mask] - mu) / sd, y[train_mask], seed=seed + k)
        acc, pred = evaluate(W, b, (X[test_mask] - mu) / sd, y[test_mask])
        oof[test_mask] = pred
        fold_accs.append(acc)
        print(f"  fold {k + 1}/{folds}: {acc:.1%} "
              f"({int(test_mask.sum())} utts, {len(groups[k])} speakers held out)")
    return float(np.mean(fold_accs)), oof, fold_accs


def print_report(y, pred, title):
    print(f"\n{title}")
    print(f"  overall accuracy: {(pred == y).mean():.1%}  (n={len(y)})")
    print("  per-class:")
    print(f"  {'emotion':<10} {'acc':>6} {'support':>8}")
    for c in range(pe.NUM_EMOTIONS):
        mask = y == c
        if not mask.any():
            continue
        acc = float((pred[mask] == c).mean())
        print(f"  {pe.EMOTION_NAMES[c]:<10} {acc:>6.1%} {int(mask.sum()):>8}")
    print("  confusion (rows=true, cols=pred):")
    header = "          " + "".join(f"{pe.EMOTION_NAMES[c][:8]:>9}" for c in range(pe.NUM_EMOTIONS))
    print(header)
    for c in range(pe.NUM_EMOTIONS):
        row = [int(((y == c) & (pred == p)).sum()) for p in range(pe.NUM_EMOTIONS)]
        print(f"  {pe.EMOTION_NAMES[c][:8]:<8}" + "".join(f"{v:>9}" for v in row))


def print_feature_importance(W, feat_std, top: int = 15):
    imp = (np.abs(W) * feat_std[None, :]).mean(axis=0)
    order = np.argsort(-imp)[:top]
    print(f"\n  top-{top} features by |W|*std (class-averaged):")
    for i in order:
        print(f"    {pe.FEATURE_NAMES[i]:<18} {imp[i]:.3f}")



# ---------------------------------------------------------------------------
# --selftest: synthetic speakers with distinct prosody profiles.
# Validates the whole pipeline (features -> CV -> training -> npz) without any
# dataset download. NOT a substitute for RAVDESS/CREMA-D training.
# ---------------------------------------------------------------------------

_SELFTEST_PROFILES = {
    # emotion_id: (f0_mean, f0_vibrato_depth, amp, syllable_rate, noise)
    0: (140.0, 10.0, 0.12, 3.5, 0.002),   # neutral
    1: (235.0, 55.0, 0.22, 5.0, 0.002),   # happy
    2: (100.0, 6.0, 0.06, 2.0, 0.001),    # sad
    3: (190.0, 35.0, 0.32, 4.5, 0.010),   # angry
    4: (265.0, 80.0, 0.18, 3.0, 0.002),   # surprise
    5: (215.0, 18.0, 0.14, 4.2, 0.004),   # fear
}


def synth_selftest_sample(emo: int, rng: np.random.Generator) -> np.ndarray:
    f0m, depth, amp, syll, noise = _SELFTEST_PROFILES[emo]
    seconds = float(rng.uniform(1.5, 3.0))
    speaker_f0 = float(rng.uniform(-20, 20))          # simulated speaker offset
    t = np.arange(int(seconds * SR)) / SR
    f0 = f0m + speaker_f0 + depth * np.sin(2 * np.pi * rng.uniform(2, 4) * t)
    phase = 2 * np.pi * np.cumsum(f0) / SR
    sig = np.zeros_like(t)
    for h, a in ((1, 1.0), (2, 0.5), (3, 0.25), (4, 0.12)):
        sig += a * np.sin(h * phase)
    env = amp * np.clip(0.55 + 0.45 * np.sin(2 * np.pi * syll * t), 0.05, 1.0)
    return (env * sig + noise * rng.standard_normal(t.shape[0])).astype(np.float32)


def run_selftest(out_path: Path, folds: int, seed: int) -> None:
    rng = np.random.default_rng(seed)
    X, y, spk = [], [], []
    n_per = 40
    speakers = [f"synthetic_{s}" for s in range(12)]
    for emo in range(pe.NUM_EMOTIONS):
        for s in range(n_per):
            speaker = speakers[s % len(speakers)]
            rng_spk = np.random.default_rng(rng.integers(0, 2**31))
            x = synth_selftest_sample(emo, rng_spk)
            for variant in (x, add_channel_noise(x, rng)):  # clean + noisy copy
                feats = pe.extract_features(variant)
                X.append(feats)
                y.append(emo)
                spk.append(speaker)
    X, y, spk = np.stack(X), np.array(y), np.array(spk)
    print(f"selftest dataset: {X.shape[0]} synthetic utterances (clean + noise-augmented), "
          f"{len(np.unique(spk))} simulated speakers")
    mean_acc, oof, _ = speaker_cv(X, y, spk, folds=folds, seed=seed)
    print_report(y, oof, "selftest speaker-grouped CV (synthetic data)")
    mu, sd = X.mean(0), np.maximum(X.std(0), 1e-3)
    W, b = train_softmax((X - mu) / sd, y, seed=seed)
    pe.save_weights(out_path, W, b, mu, sd)
    print(f"\n[ok] selftest weights written: {out_path} "
          f"({out_path.stat().st_size} bytes), CV acc {mean_acc:.1%}")
    if mean_acc < 0.55:
        sys.exit("selftest CV accuracy suspiciously low - the pipeline is broken")

    # Channel-robustness regression guard: synthetic profiles under 12 dB noise
    # must not collapse to one class (the all-angry bug this augmentation fixes).
    rng_guard = np.random.default_rng(seed + 99)
    clf = pe.ProsodyEmotionClassifier(weights_path=out_path, verbose=False)
    preds = [clf.predict(add_channel_noise(synth_selftest_sample(emo, rng_guard),
                                           rng_guard, snr_db=12.0))[0]
             for emo in range(pe.NUM_EMOTIONS)]
    n_ok = sum(int(p == e) for e, p in enumerate(preds))
    print(f"[ok] noise-robustness guard: {n_ok}/{pe.NUM_EMOTIONS} profiles correct at "
          f"12 dB SNR: {[pe.EMOTION_NAMES[p] for p in preds]}")
    if n_ok < pe.NUM_EMOTIONS - 1:
        sys.exit("noise-robustness guard failed: predictions collapse under channel noise")

    # Edge-silence guard: profiles padded with 2.4 s of edge silence must still
    # classify correctly (studio data is trimmed; real recordings are not).
    preds_pad = []
    for emo in range(pe.NUM_EMOTIONS):
        x = synth_selftest_sample(emo, rng_guard)
        pad = np.concatenate([np.zeros(int(0.6 * SR), np.float32), x,
                              np.zeros(int(1.8 * SR), np.float32)]).astype(np.float32)
        preds_pad.append(clf.predict(pad)[0])
    n_ok = sum(int(p == e) for e, p in enumerate(preds_pad))
    print(f"[ok] edge-silence guard: {n_ok}/{pe.NUM_EMOTIONS} profiles correct with "
          f"2.4 s of edge silence: {[pe.EMOTION_NAMES[p] for p in preds_pad]}")
    if n_ok < pe.NUM_EMOTIONS - 1:
        sys.exit("edge-silence guard failed: predictions change under edge padding")


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ravdess_dir", default="", help="RAVDESS speech wav root")
    ap.add_argument("--cremad_dir", default="", help="CREMA-D wav root (optional)")
    ap.add_argument("--out", default=str(ROOT / "IMTalker" / "checkpoints" / "prosody_emotion.npz"))
    ap.add_argument("--augment", action="store_true",
                    help="5x data: gain {0.7,1.3} + tape speed {0.9,1.1}")
    ap.add_argument("--noise_aug", action=argparse.BooleanOptionalAction, default=True,
                    help="add a noisy copy of every utterance (default: on; the "
                         "channel-robustness fix for non-studio audio)")
    ap.add_argument("--extra_dir", default="",
                    help="optional user folder with <emotion>-*.wav files "
                         "(angry/fear/happy/sad/neutral/surprise); added x%d" % EXTRA_REPEAT)
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--selftest", action="store_true",
                    help="train on synthetic speakers (no dataset needed; pipeline check)")
    args = ap.parse_args()

    out_path = Path(args.out)
    if args.selftest:
        run_selftest(out_path, args.folds, args.seed)
        return

    samples = []
    if args.ravdess_dir:
        found = scan_ravdess(Path(args.ravdess_dir))
        print(f"RAVDESS: {len(found)} usable clips under {args.ravdess_dir}")
        samples += found
    if args.cremad_dir:
        found = scan_cremad(Path(args.cremad_dir))
        print(f"CREMA-D: {len(found)} usable clips under {args.cremad_dir}")
        samples += found
    extra_names = []
    if args.extra_dir:
        found = scan_extra(Path(args.extra_dir))
        if found:
            emos = sorted({pe.EMOTION_NAMES[e] for _, e, _ in found})
            print(f"extra audio: {len(found)} usable clips under {args.extra_dir} "
                  f"({', '.join(emos)}), replicated x{EXTRA_REPEAT}")
            print("             note: these files are IN the training set - testing on the")
            print("             same folder afterwards measures training data, not generalisation")
            samples += found * EXTRA_REPEAT
            extra_names = sorted({wav.name for wav, _, _ in found})
        else:
            print(f"[warn] --extra_dir given but no <emotion>-*.wav files found under {args.extra_dir}")
    if not samples:
        sys.exit("no clips found - pass --ravdess_dir/--cremad_dir or use --selftest")

    aug_desc = []
    if args.augment:
        aug_desc.append("5x gain/speed")
    if args.noise_aug:
        aug_desc.append("+1 noisy copy each (6-30 dB SNR)")
    print(f"extracting features ({', '.join(aug_desc) or 'no augmentation'})...")
    X, y, speakers = build_dataset(samples, augment=args.augment,
                                   noise_aug=args.noise_aug,
                                   rng=np.random.default_rng(args.seed))
    print(f"dataset: {X.shape[0]} utterances, {len(np.unique(speakers))} speakers, "
          f"class counts {np.bincount(y, minlength=pe.NUM_EMOTIONS).tolist()}")

    print(f"\nspeaker-grouped {args.folds}-fold CV:")
    mean_acc, oof, fold_accs = speaker_cv(X, y, speakers, folds=args.folds, seed=args.seed)
    print_report(y, oof, "speaker-independent CV (held-out speakers = unseen voices)")
    if extra_names:
        user_mask = speakers == "user_extra"
        # user files are their own speaker group: one fold held them ALL out,
        # so their out-of-fold predictions are an honest generalisation estimate
        print(f"\n  user extra files (held out as their own speaker group): "
              f"{(oof[user_mask] == y[user_mask]).mean():.1%} honest accuracy on your audio "
              f"(n={int(user_mask.sum())})")

    # final model: all data
    mu, sd = X.mean(0), np.maximum(X.std(0), 1e-3)
    W, b = train_softmax((X - mu) / sd, y, seed=args.seed)
    train_acc, _ = evaluate(W, b, (X - mu) / sd, y)
    print(f"\nfinal model train accuracy (all data): {train_acc:.1%}")
    print_feature_importance(W, sd)

    pe.save_weights(out_path, W, b, mu, sd, extra_files=extra_names)
    print(f"\n[ok] weights written: {out_path} ({out_path.stat().st_size} bytes)")

    # quick latency check with the freshly trained model
    clf = pe.ProsodyEmotionClassifier(weights_path=out_path, verbose=False)
    probe = np.clip(0.15 * np.sin(2 * np.pi * 220 * np.arange(3 * SR) / SR), -1, 1).astype(np.float32)
    for _ in range(3):
        clf.predict(probe)
    t0 = time.perf_counter()
    for _ in range(20):
        clf.predict(probe)
    print(f"[ok] inference latency: {(time.perf_counter() - t0) / 20 * 1e3:.1f} ms/utterance (CPU)")
    print("\nnext step: re-run scripts/switch_ser_to_prosody.py to embed these "
          "weights into runpod_emotion_streaming.ipynb before deploying.")


if __name__ == "__main__":
    main()

''',
}


def have_all(root):
    return all((Path(root) / rel).is_file() for rel in REQUIRED_FILES)


if have_all(PROJECT_ROOT):
    print(f"[skip] repo already present with prosody files: {PROJECT_ROOT}")
else:
    if not Path(PROJECT_ROOT).exists():
        print(f"[clone] {REPO_URL} -> {PROJECT_ROOT}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_ROOT], check=True)
    if have_all(PROJECT_ROOT):
        print(f"[ok] cloned repo contains the prosody files: {PROJECT_ROOT}")
    else:
        print("[warn] the repo checkout is missing the prosody files - writing embedded copies")
        print("       (push your local workspace to the repo to make this unnecessary)")
        for rel, payload in EMBEDDED.items():
            dest = Path(PROJECT_ROOT) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(payload, encoding="utf-8")
            print(f"[written] {dest} ({len(payload)} chars, embedded copy)")

assert have_all(PROJECT_ROOT), f"prosody files missing after setup: {REQUIRED_FILES}"
print(f"[ok] project ready: {PROJECT_ROOT}")


## Step 2 — Python dependencies

Only tiny wheels: `numpy` (features + training) and `soundfile` (WAV loading;
`scipy` is an automatic fallback loader when absent). No torch, no
transformers, no GPU.


In [ ]:
import importlib
import subprocess
import sys


def ensure(module, pip_name=None):
    try:
        importlib.import_module(module)
        print(f"[ok] {module}")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)
        print(f"[installed] {pip_name or module}")


ensure("numpy")
ensure("soundfile")
try:
    importlib.import_module("scipy")
    print("[ok] scipy (optional fallback loader available)")
except ImportError:
    print("[info] scipy not installed - soundfile will be used (that is fine)")


## Step 3 — Download RAVDESS (required)

The speech portion of the Ryerson Audio-Visual Database of Emotional Speech
and Song: `Audio_Speech_Actors_01-24.zip` (~215 MB, 1,440 clips, 24 actors,
48 kHz 16-bit WAV) from Zenodo record 1188976.

Filename convention: `03-01-{emotion}-{intensity}-{statement}-{rep}-{actor}.wav`
with emotion codes 01 neutral, 02 calm, 03 happy, 04 sad, 05 angry, 06 fearful,
07 disgust, 08 surprised. Training maps **neutral+calm → 0**, happy → 1,
sad → 2, angry → 3, **surprised → 4**, fearful → 5 (disgust is skipped: the
avatar has no disgust expression).

License: CC BY-NC-SA 4.0 — non-commercial research use; cite Livingstone &
Russo (2018), PLoS ONE 13(5): e0196391.

Idempotent: re-running the cell skips an already-extracted dataset.


In [ ]:
import time
import urllib.request
import zipfile
from pathlib import Path

DATA_ROOT = Path(DATA_DIR)
RAVDESS_URLS = [
    "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1",
    "https://zenodo.org/api/records/1188976/files/Audio_Speech_Actors_01-24.zip/content",
]
RAVDESS_ZIP = DATA_ROOT / "Audio_Speech_Actors_01-24.zip"
RAVDESS_DIR = DATA_ROOT / "ravdess"


def download(urls, dest, min_bytes=1_000_000, retries=3):
    dest.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for attempt in range(1, retries + 1):
        url = urls[(attempt - 1) % len(urls)]
        try:
            print(f"[download] {url} (attempt {attempt}/{retries})")
            t0 = time.time()
            with urllib.request.urlopen(url, timeout=120) as resp, open(dest, "wb") as fh:
                total = 0
                while True:
                    chunk = resp.read(1 << 20)
                    if not chunk:
                        break
                    fh.write(chunk)
                    total += len(chunk)
            if total < min_bytes:
                raise IOError(f"downloaded only {total} bytes")
            print(f"[ok] {dest.name}: {total / 1e6:.1f} MB in {time.time() - t0:.0f}s")
            return
        except Exception as exc:
            print(f"[warn] attempt {attempt} failed: {exc!r}")
            last = exc
            if dest.exists():
                dest.unlink()
    raise last


if (RAVDESS_DIR / ".done").is_file():
    print(f"[skip] RAVDESS already extracted: {RAVDESS_DIR}")
else:
    download(RAVDESS_URLS, RAVDESS_ZIP, min_bytes=200_000_000)
    with zipfile.ZipFile(RAVDESS_ZIP) as zf:
        bad = zf.testzip()
        assert bad is None, f"corrupt member in {RAVDESS_ZIP}: {bad}"
        print(f"[extract] {RAVDESS_ZIP.name} -> {RAVDESS_DIR} (~1 min)")
        RAVDESS_DIR.mkdir(parents=True, exist_ok=True)
        zf.extractall(RAVDESS_DIR)
    wavs = sorted(RAVDESS_DIR.rglob("*.wav"))
    assert len(wavs) == 1440, f"expected 1440 RAVDESS clips, found {len(wavs)}"
    (RAVDESS_DIR / ".done").write_text("ok", encoding="utf-8")

print(f"[ok] RAVDESS ready: {len(list(RAVDESS_DIR.rglob('*.wav')))} clips in {RAVDESS_DIR}")


## Step 4 — Download CREMA-D (optional, recommended)

7,442 clips from 91 actors (diverse ages/accents) — the speaker diversity that
RAVDESS (24 actors) lacks. Clips with `XX` (unspecified) intensity are skipped;
the rest map onto the same 6 classes.

The wavs live in **git-LFS** inside the CREMA-D GitHub repo (a plain clone only
yields 130-byte pointer files), so this cell installs `git-lfs` (via apt when
missing) and does a sparse blobless clone + `git lfs pull` of `AudioWAV/` only
(~600 MB). If anything fails — e.g. the repo's public LFS bandwidth quota —
the cell prints a warning and training continues **RAVDESS-only**. Set
`USE_CREMAD = False` to skip deliberately.

Idempotent: re-running skips a completed download.


In [ ]:
import shutil
import subprocess
from pathlib import Path

CREMAD_DIR = DATA_ROOT / "cremad"
CREMAD_AUDIO_DIR = CREMAD_DIR / "AudioWAV"
cremad_ok = False


def _run(cmd, **kw):
    proc = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if proc.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed: {proc.stderr[-400:]}")
    return proc


if not USE_CREMAD:
    print("[skip] USE_CREMAD = False")
elif (CREMAD_DIR / ".done").is_file():
    cremad_ok = True
    print(f"[skip] CREMA-D already downloaded: {CREMAD_AUDIO_DIR}")
else:
    try:
        try:
            _run(["git", "lfs", "version"])
        except Exception:
            print("[setup] git-lfs missing - installing via apt-get (RunPod pods run as root)")
            try:
                _run(["apt-get", "update", "-qq"], timeout=300)
            except Exception as exc:
                print(f"[warn] apt-get update failed ({exc!r}); trying install anyway")
            _run(["apt-get", "install", "-y", "git-lfs"], timeout=600)
        _run(["git", "lfs", "install"])
        if CREMAD_DIR.exists():
            shutil.rmtree(CREMAD_DIR)
        _run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
              "https://github.com/CheyneyComputerScience/CREMA-D.git", str(CREMAD_DIR)], timeout=600)
        _run(["git", "-C", str(CREMAD_DIR), "sparse-checkout", "set", "AudioWAV"], timeout=600)
        print("[download] git lfs pull (AudioWAV, ~600 MB) - this takes a few minutes ...")
        _run(["git", "-C", str(CREMAD_DIR), "lfs", "pull"], timeout=5400)
        wavs = sorted(CREMAD_AUDIO_DIR.glob("*.wav"))
        assert len(wavs) >= 7000, f"expected ~7442 CREMA-D clips, found {len(wavs)}"
        small = [w for w in wavs[:50] if w.stat().st_size < 2048]
        assert not small, "files look like git-LFS pointers (lfs pull did not fetch content)"
        (CREMAD_DIR / ".done").write_text("ok", encoding="utf-8")
        cremad_ok = True
        print(f"[ok] CREMA-D ready: {len(wavs)} clips in {CREMAD_AUDIO_DIR}")
    except Exception as exc:
        print(f"[warn] CREMA-D download failed: {exc!r}")
        print("[warn] continuing with RAVDESS only (1,440 clips, 24 speakers) -")
        print("       the model still trains; extra speaker diversity would improve it")


## Step 5 — Dataset audit

Counts the usable clips per avatar emotion class for each dataset, using the
trainer's own scanners — so what you see is exactly what training will use.


In [ ]:
import importlib.util
from collections import Counter
from pathlib import Path

# Import the trainer module by path (scripts/ is not a package).
_spec = importlib.util.spec_from_file_location(
    "train_prosody_emotion", Path(PROJECT_ROOT) / "scripts" / "train_prosody_emotion.py")
trainer = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(trainer)

NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprised", 5: "fearful"}


def audit(label, files):
    # scanner rows are (wav_path, emotion_id, speaker_id)
    counts = Counter(fid for _, fid, _spk in files)
    total = max(1, sum(counts.values()))
    print(f"{label}: {total} clips")
    for fid in range(6):
        n = counts.get(fid, 0)
        print(f"  id={fid} {NAMES[fid]:<9} {n:>5} clips ({100 * n / total:.1f}%)")


ravdess_files = trainer.scan_ravdess(str(RAVDESS_DIR))
audit("RAVDESS (speech, neutral+calm merged)", ravdess_files)
cremad_files = []
if cremad_ok:
    cremad_files = trainer.scan_cremad(str(CREMAD_AUDIO_DIR))
    audit("CREMA-D (XX intensity skipped)", cremad_files)
total = len(ravdess_files) + len(cremad_files)
assert total > 0, "no training clips found - check the download cells above"
print(f"[ok] total training clips: {total}")


## Step 6 — Train

Runs `scripts/train_prosody_emotion.py`:

- 56-dim prosodic features per clip (F0 stats/contour, voiced-time RMS stats,
  pauses / speech rate, spectral flux / centroid, voiced ZCR, coarse formant
  proxies) — the identical code path the live avatar uses at inference.
- **Noise augmentation (always on)**: every training clip also appears once
  with added white/pink noise at a random 6–30 dB SNR. RAVDESS/CREMA-D are
  studio recordings; without this the model learns "bright/noisy spectrum =
  angry" and collapses *every real-world recording to angry*. This is the
  channel-robustness fix.
- **Your own audio** (`EXTRA_AUDIO_DIR`): files named `<emotion>-*.wav`
  (angry/fear/happy/sad/neutral/surprise) are added to training ×10. They form
  their own CV speaker group, so the report shows an honest held-out accuracy
  on *your* audio. Set to `""` for pure dataset training.
- Standardize → numpy softmax regression (full-batch L-BFGS).
- **Speaker-grouped k-fold CV** (no speaker appears in both train and test) →
  an honest generalization estimate, then a final refit on 100% of the data.
- Writes `prosody_emotion.npz` (~7 KB: weights, bias, mu, sd, meta).

Expected duration: ~15–30 min with CREMA-D (CPU-bound feature extraction),
a few minutes RAVDESS-only.


In [ ]:
import subprocess
import sys
import time
from pathlib import Path

WEIGHTS_PATH = Path(PROJECT_ROOT) / "IMTalker" / "checkpoints" / "prosody_emotion.npz"
WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Your own labeled audio (files named <emotion>-*.wav) joins the training set
# x10 - adapts the model to your voice/room. Set to "" for pure dataset training.
EXTRA_AUDIO_DIR = "/workspace/audio"

cmd = [
    sys.executable, str(Path(PROJECT_ROOT) / "scripts" / "train_prosody_emotion.py"),
    "--ravdess_dir", str(RAVDESS_DIR),
    "--out", str(WEIGHTS_PATH),
    "--folds", str(FOLDS), "--seed", str(SEED),
]
if cremad_ok:
    cmd += ["--cremad_dir", str(CREMAD_AUDIO_DIR)]
if AUGMENT:
    cmd += ["--augment"]
if EXTRA_AUDIO_DIR:
    if Path(EXTRA_AUDIO_DIR).is_dir():
        cmd += ["--extra_dir", EXTRA_AUDIO_DIR]
    else:
        print(f"[skip] EXTRA_AUDIO_DIR does not exist: {EXTRA_AUDIO_DIR}")
print("[run]", " ".join(cmd))
t0 = time.time()
proc = subprocess.run(cmd)
assert proc.returncode == 0, "training failed - see the output above"
print(f"[ok] training finished in {(time.time() - t0) / 60:.1f} min")
assert WEIGHTS_PATH.is_file() and WEIGHTS_PATH.stat().st_size > 1000
print(f"[ok] weights: {WEIGHTS_PATH} ({WEIGHTS_PATH.stat().st_size:,} bytes)")


## Step 7 — Validate + hand off

Loads the trained weights through `ProsodyEmotionClassifier` (the inference
entry point used by the avatar), classifies one demo clip per emotion, measures
per-utterance latency, and copies the weights to `WEIGHTS_OUT` — the exact
path the streaming notebook's `PROSODY_WEIGHTS_FILE` parameter points to.


In [ ]:
import shutil
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(PROJECT_ROOT) / "IMTalker"))
import prosody_emotion as pe

clf = pe.ProsodyEmotionClassifier(weights_path=str(WEIGHTS_PATH))
assert clf.backend == "softmax", f"expected the trained softmax backend, got {clf.backend!r}"
print(f"[ok] loaded trained classifier: backend={clf.backend}")

# One demo clip per emotion (CREMA-D preferred for speaker diversity; RAVDESS
# fills in). Single clips are a sanity check, not an accuracy estimate.
demo = {}
for path, fid, _spk in (trainer.scan_cremad(str(CREMAD_AUDIO_DIR)) if cremad_ok else []):
    demo.setdefault(fid, path)
for path, fid, _spk in trainer.scan_ravdess(str(RAVDESS_DIR)):
    demo.setdefault(fid, path)

n_correct = 0
for fid in sorted(demo):
    wav = trainer.load_wav(demo[fid])  # resampled to 24 kHz, the classifier's input rate
    y, label, conf = clf.predict(wav)
    ok = "OK  " if y == fid else "MISS"
    n_correct += (y == fid)
    print(f"  [{ok}] expected id={fid} {NAMES[fid]:<9} -> predicted id={y} "
          f"{label:<10} conf={conf:.2f}  ({Path(demo[fid]).name})")
print(f"[info] demo sanity check: {n_correct}/{len(demo)} correct (single clips)")

# Latency: the live avatar runs this on a daemon CPU thread per utterance.
wav = trainer.load_wav(demo[sorted(demo)[0]])
for _ in range(3):
    clf.predict(wav)  # warm-up
t0 = time.perf_counter()
reps = 20
for _ in range(reps):
    clf.predict(wav)
ms = (time.perf_counter() - t0) / reps * 1000
print(f"[ok] latency: {ms:.1f} ms/utterance on this CPU")

# Hand off: the streaming notebook picks the weights up from WEIGHTS_OUT.
shutil.copyfile(WEIGHTS_PATH, WEIGHTS_OUT)
print(f"[ok] copied -> {WEIGHTS_OUT}")
print("[next] run runpod_emotion_streaming_prosody.ipynb on this pod - it finds")
print("       the weights at PROSODY_WEIGHTS_FILE automatically. If this notebook")
print("       ran elsewhere, upload the .npz to the pod at /workspace/prosody_emotion.npz.")


## Step 8 — Test on your own audio folder (optional, re-runnable)

Point `TEST_AUDIO_DIR` at any folder of audio clips and re-run the cell below
as often as you like — no retraining needed. The trained `.npz` is loaded once
and every file is classified with its **confidence score** plus the top-3
probability breakdown.

Recursively scans `.wav .flac .ogg .mp3 .m4a` (mp3/m4a support depends on the
soundfile build; unreadable files are reported, not fatal). Silent or
too-short clips are skipped with a note. The cell reloads the trainer helpers
and the trained weights if they are not in memory, so it also works after a
kernel restart.

Two robustness details (mirroring the live avatar):

- **Long files are segmented** into ~4 s chunks — the model was trained on
  1–10 s utterances, and the live avatar classifies VAD segments of the same
  size. Per-chunk votes are shown and the final prediction averages the
  chunk probabilities.
- **Out-of-distribution warnings**: if a file's features sit far outside the
  RAVDESS training distribution (max |z| > 5, e.g. another language, shouting,
  singing), the cell prints which features are off-scale. Predictions there
  are clipped to the |z| ≤ 5 boundary instead of saturating — treat their
  confidence as unreliable.
- **Edge silence is trimmed** (VAD-style) before classification — studio
  training clips have none, and leftover silence corrupts energy/pause
  features. Trimmed seconds show as `6.4s speech (of 8.1s)`.


In [ ]:
# --- Test the trained classifier on your own audio folder ---------------------
TEST_AUDIO_DIR = "/workspace/audio"  # <-- edit me, then re-run this cell
AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".m4a"}
SEGMENT_S = 4.0    # classify long files in utterance-sized chunks (like the live VAD)
MAX_FILE_S = 6.0   # files longer than this are segmented and their votes averaged
OOD_Z = 5.0        # |z| beyond this = feature far outside the training distribution

import time
from collections import Counter
from pathlib import Path

import numpy as np

# Re-runnable standalone (e.g. after a kernel restart): reload helpers/weights.
if "WEIGHTS_PATH" not in dir():
    WEIGHTS_PATH = Path(PROJECT_ROOT) / "IMTalker" / "checkpoints" / "prosody_emotion.npz"
if "trainer" not in dir():
    import importlib.util
    _spec = importlib.util.spec_from_file_location(
        "train_prosody_emotion", Path(PROJECT_ROOT) / "scripts" / "train_prosody_emotion.py")
    trainer = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(trainer)
if "pe" not in dir():
    import sys
    sys.path.insert(0, str(Path(PROJECT_ROOT) / "IMTalker"))
    import prosody_emotion as pe
if "clf" not in dir() or getattr(clf, "backend", None) != "softmax":
    clf = pe.ProsodyEmotionClassifier(weights_path=str(WEIGHTS_PATH))
    assert clf.backend == "softmax", "trained weights not loaded - run the training cell first"

# files that were included in training via --extra_dir (metadata in the npz) -
# predictions on them are training-set results, not generalisation evidence
_trained_on = set()
if WEIGHTS_PATH.is_file():
    with np.load(WEIGHTS_PATH) as _w:
        if "extra_files" in _w.files:
            _trained_on = {str(x) for x in _w["extra_files"]}


def classify_wav(wav):
    # Classify a waveform: edge silence is trimmed first (VAD-style, mirroring
    # the live avatar), files longer than MAX_FILE_S are split into SEGMENT_S
    # chunks and aggregated by mean probability. Also returns the max |z|
    # score and the most out-of-distribution features seen, for diagnostics.
    wav = pe.trim_edges(wav)
    if len(wav) < pe.FRAME_LEN * 2:
        return [], Counter(), 0.0, []
    seg = int(SEGMENT_S * pe.INPUT_SR)
    if len(wav) <= int(MAX_FILE_S * pe.INPUT_SR):
        chunks = [wav]
    else:
        chunks = [wav[i:i + seg] for i in range(0, len(wav), seg)]
        if len(chunks) > 1 and len(chunks[-1]) < seg // 4:
            chunks[-2] = np.concatenate([chunks[-2], chunks[-1]])
            chunks.pop()
    mu, sd = clf.model.feat_mean, clf.model.feat_std
    probs_list, votes, max_z, ood = [], Counter(), 0.0, []
    for c in chunks:
        feats = pe.extract_features(c)
        if not feats.any():
            continue
        z = np.nan_to_num((feats - mu) / sd, nan=0.0, posinf=0.0, neginf=0.0)
        az = np.abs(z)
        if float(az.max()) > max_z:
            max_z = float(az.max())
            ood = [(pe.FEATURE_NAMES[i], float(z[i])) for i in np.argsort(-az)[:3]]
        p = clf.model.predict_proba(feats)  # clips z at +/-pe.Z_CLIP internally
        probs_list.append(p)
        votes[int(np.argmax(p))] += 1
    return probs_list, votes, max_z, ood


test_dir = Path(TEST_AUDIO_DIR)
if not test_dir.is_dir():
    print(f"[skip] TEST_AUDIO_DIR does not exist: {test_dir}")
    print("       upload audio there (or edit the path above) and re-run this cell")
else:
    files = sorted(p for p in test_dir.rglob("*") if p.suffix.lower() in AUDIO_EXTS)
    if not files:
        print(f"[warn] no audio files ({', '.join(sorted(AUDIO_EXTS))}) found under {test_dir}")
    else:
        print(f"[test] classifying {len(files)} file(s) from {test_dir} "
              f"(weights: {clf.weights_path})\n")
        print(f"{'file':<50} {'predicted':<14} {'conf':>5}  top-3  [details]")
        print("-" * 110)
        counts = Counter()
        n_skip = n_err = n_ood = n_extra = 0
        t0 = time.perf_counter()
        for p in files:
            try:
                wav = trainer.load_wav(p)  # mono float32 @ 24 kHz
                dur_raw = len(wav) / pe.INPUT_SR
                wav = pe.trim_edges(wav)   # idempotent; classify_wav re-checks
                probs_list, votes, max_z, ood = classify_wav(wav)
                if not probs_list:
                    n_skip += 1
                    print(f"{p.name[:50]:<50} {'skipped':<14} {'--':>5}  silent or too short")
                    continue
                probs = np.mean(probs_list, axis=0)
                order = np.argsort(probs)[::-1]
                y = int(order[0])
                label = pe.EMOTION_NAMES[y]
                conf = float(probs[y])
                counts[label] += 1
                top3 = ", ".join(f"{pe.EMOTION_NAMES[i]} {probs[i]:.2f}" for i in order[:3])
                dur = len(wav) / pe.INPUT_SR
                detail = (f"{dur:.1f}s speech (of {dur_raw:.1f}s)" if dur_raw - dur > 0.2
                          else f"{dur:.1f}s")
                if len(probs_list) > 1:
                    tally = ", ".join(f"{pe.EMOTION_NAMES[k]}x{v}" for k, v in votes.most_common())
                    detail += f", {len(probs_list)} chunks: {tally}"
                if p.name in _trained_on:
                    n_extra += 1
                    detail += ", in training"
                print(f"{p.name[:50]:<50} {label + f' ({y})':<14} {conf:5.2f}  {top3}  [{detail}]")
                if max_z > OOD_Z:
                    n_ood += 1
                    worst = ", ".join(f"{n} z={v:+.1f}" for n, v in ood if abs(v) > OOD_Z)
                    print(f"{'':<52}[warn] out-of-distribution (max|z|={max_z:.1f}): {worst}")
            except Exception as exc:
                n_err += 1
                print(f"{p.name[:50]:<50} ERROR: {exc}")
        dt = time.perf_counter() - t0
        print("-" * 110)
        summary = ", ".join(f"{k} {v}" for k, v in sorted(counts.items())) or "none"
        print(f"[summary] {sum(counts.values())}/{len(files)} classified: {summary}")
        if n_skip or n_err:
            print(f"[info] {n_skip} skipped (silent/short), {n_err} unreadable")
        if n_ood:
            print(f"[warn] {n_ood} file(s) were far outside the RAVDESS training distribution")
            print(f"       (max|z| > {OOD_Z:.0f}); their confidence is unreliable - predictions")
            print(f"       there are clipped to the |z| <= {pe.Z_CLIP:.0f} boundary, not saturated.")
        if n_extra:
            print(f"[info] {n_extra} file(s) marked 'in training' were part of --extra_dir")
            print(f"       training - high accuracy on them is expected, not generalisation")
        if counts:
            print(f"[info] avg {dt / sum(counts.values()) * 1000:.1f} ms/file")


## Done — next steps

1. Run `runpod_emotion_streaming_prosody.ipynb` on the **same pod** (shared
   `/workspace`): its Step 12 copies `/workspace/prosody_emotion.npz` into
   `IMTalker/checkpoints/` and the SER smoke test prints
   `prosody backend = softmax`.
2. Trained on another machine? Upload `prosody_emotion.npz` (~7 KB) to the pod
   at `/workspace/prosody_emotion.npz` (Jupyter file upload) before Step 12.
3. No weights file at all → the avatar still runs on the built-in rule-based
   prosody fallback; retraining is optional.
4. **Step 8** above batch-tests any audio folder (`TEST_AUDIO_DIR`) against
   the trained weights with confidence scores — re-runnable at any time.

**Rollback:** delete `IMTalker/checkpoints/prosody_emotion.npz` (rule fallback)
or set `SER_MODEL` to any HF backend — every previous SER model remains
available.

**Retraining tips:** more data helps most (add IEMOCAP/MSP via the same
filename→label pattern); `AUGMENT = False` for a quick iteration; `FOLDS`
trades CV runtime vs. estimate stability.

**Citations:** RAVDESS — Livingstone & Russo (2018), PLoS ONE 13(5): e0196391
(CC BY-NC-SA 4.0). CREMA-D — Cao et al. (2014), IEEE Trans. Affective
Computing 5(4) (see the CREMA-D repo for its license terms).
